# Detection: train Channel A and its ablations

Trains the span head and relation-slot head on the LettuceDetect encoder chosen in the phase-1 pilot (set CHECKPOINT), then the no-slot ablation, and evaluates both against Channel B on 2,000 development items. If XBRL-mined examples exist (runs/detector/xbrl.jsonl from `faithguard xbrl-mine`), the no-XBRL ablation runs too. Expect roughly 3-5 GPU hours.

**On Kaggle:** Create → New Notebook → File → Import Notebook → this file. Settings: *Accelerator* **GPU T4 x2**, *Internet* **on**.
For long jobs use **Save Version → Save & Run All (Commit)**: it runs in the background for up to 12 hours and keeps everything in `/kaggle/working/outputs`.
To continue in a later notebook, add this notebook's output as input (*Add Input → Your Work*).

**Afterwards:** download `outputs/*/results.json` and `report.md` files into `runs/` in the repository, and log the session's GPU hours in `logs/gpu-hours.csv`. Checkpoints are saved regularly; re-running the same cell resumes from the latest one.

In [ ]:
# Setup: clone FaithGuard, install it with the pinned GPU packages, fetch the data.
import os, subprocess, sys
WORK = '/kaggle/working' if os.path.exists('/kaggle') else '/content'
os.chdir(WORK)
if not os.path.exists('faithguard'):
    subprocess.check_call(['git', 'clone', '--depth', '1', 'https://github.com/SharafThawfeek/FaithGurd-new.git', 'faithguard'])
os.chdir('faithguard')
REQUIREMENTS = ['transformers==5.19.0', 'peft==0.21.2', 'accelerate==1.15.0', 'bitsandbytes==0.50.2', 'huggingface_hub==1.33.0', 'datasets==5.1.0', 'lettucedetect==0.2.3']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *REQUIREMENTS])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'])
OUT = os.path.join(WORK, 'outputs'); os.makedirs(OUT, exist_ok=True)
!faithguard data download all
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Training data (controlled track + RAGTruth, + XBRL if mined)

In [ ]:
CHECKPOINT = 'KRLabsOrg/lettucedect-v2-mmbert-base'  # or KRLabsOrg/lettucedect-large-modernbert-en-v1
!faithguard detector-data

## Channel A, full model

In [ ]:
!python -m faithguard.train.detector --data runs/detector/train.jsonl --checkpoint $CHECKPOINT --out {OUT}/channel-a

## Ablation: without the slot head

In [ ]:
!python -m faithguard.train.detector --data runs/detector/train.jsonl --checkpoint $CHECKPOINT --no-slot --out {OUT}/channel-a-noslot

## Ablation: without XBRL negatives (only if XBRL examples were mined)

In [ ]:
import os
if os.path.exists('runs/detector/xbrl.jsonl'):
    !python -m faithguard.train.detector --data runs/detector/train.jsonl --checkpoint $CHECKPOINT --exclude xbrl --out {OUT}/channel-a-noxbrl

## Evaluate (also writes per-claim rows for the A+B fusion)

In [ ]:
!python -m faithguard.train.detector_eval --model-dir {OUT}/channel-a/model --out {OUT}/eval-channel-a
!python -m faithguard.train.detector_eval --model-dir {OUT}/channel-a-noslot/model --out {OUT}/eval-channel-a-noslot

## Results

In [ ]:
import glob, json
for path in sorted(glob.glob(OUT + '/**/results.json', recursive=True)):
    print(path)
    print(json.dumps({k: v for k, v in json.load(open(path)).items() if k in ('summary', 'channel_a', 'channel_b', 'auroc', 'false_alarm_clean', 'wrong_context_recall')}, indent=1)[:1500])